# 22. SVR 0.1505가 진짜 신호인가 - train 내부 근접쌍 GroupKFold 검증

21번 노트북에서 train-test 간 근접 복제쌍(지터만 다르고 나머지 동일)이 존재한다는 걸 확인했다.
test는 절대 건드리지 않고, **train 안에서만** 같은 근접쌍 구조(774쌍)를 찾아서
일반 KFold와 GroupKFold(같은 쌍은 같은 폴드)의 SVR 점수를 비교한다.

## 결과 요약

**GroupKFold MAE(0.2499)가 상수 예측 MAE(0.2493)와 사실상 동일하다.**
즉 SVR의 0.1505라는 점수는 진짜 예측력이 아니라, train 안에 흩어진 근접쌍이 표준
KFold에서 서로 다른 폴드로 갈라져 생기는 암기 효과였다. LGBM에서 확인된 것과
동일한 현상이 SVR에서도 그대로 재현된다.

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.svm import SVR
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42
CAT = ['gender', 'activity', 'smoke_status', 'medical_history',
       'family_medical_history', 'sleep_pattern', 'edu_level']
NUM = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
       'diastolic_blood_pressure', 'glucose', 'bone_density']
TH = 0.45  # 21번 노트북에서 검증된 안정 임계값

train = pd.read_csv('../data/train.csv')
train = train.drop_duplicates(subset=[c for c in train.columns if c != 'ID']).reset_index(drop=True)
train = train.fillna('Unknown')
train['bmi'] = train['weight'].astype(float) / ((train['height'].astype(float) / 100) ** 2)
y = train['stress_score'].values
print(train.shape)

(2994, 19)


In [2]:
def pair_labels(df, cat_cols, num_cols, th):
    """범주형 블로킹 + 숫자 체비셰프 거리로 연결 요소(근접쌍) 라벨 반환. test는 전혀 안 씀."""
    sig = df[cat_cols].astype(str).agg('|'.join, axis=1).values
    V = df[num_cols].values.astype(float)
    scale = V.std(0)
    rows, cols = [], []
    order = np.argsort(sig, kind='stable')
    starts = np.flatnonzero(np.r_[True, sig[order][1:] != sig[order][:-1]])
    for s, e in zip(starts, np.r_[starts[1:], len(order)]):
        blk = order[s:e]
        if len(blk) < 2:
            continue
        d = np.abs((V[blk][:, None, :] - V[blk][None, :, :]) / scale).max(-1)
        a, b = np.nonzero(np.triu(d <= th, k=1))
        rows.extend(blk[a]); cols.extend(blk[b])
    g = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(df),) * 2)
    return connected_components(g, directed=False)[1]


groups = pair_labels(train, CAT, NUM, TH)
sizes = pd.Series(groups).value_counts()
print('그룹 크기 분포:', sizes.value_counts().sort_index().to_dict())
print(f'근접쌍(크기2) 그룹 수: {(sizes == 2).sum()}  (21번 노트북에서 확인된 train-train 774쌍과 비교)')

그룹 크기 분포: {1: 1458, 2: 768}
근접쌍(크기2) 그룹 수: 768  (21번 노트북에서 확인된 train-train 774쌍과 비교)


In [3]:
RAW14_BMI = ['gender', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
             'diastolic_blood_pressure', 'glucose', 'bone_density', 'activity',
             'smoke_status', 'medical_history', 'family_medical_history',
             'sleep_pattern', 'edu_level', 'bmi']

X = train[RAW14_BMI].copy()
for c in CAT:
    dummies = pd.get_dummies(X[c], prefix=c, dtype=int)
    X = pd.concat([X.drop(columns=[c]), dummies], axis=1)

# 20번 노트북에서 hyperopt로 찾은 최적값 그대로 재사용
SVR_C, SVR_GAMMA = 3.8944338291361977, 2.495273322374727

def make_svr():
    return make_pipeline(
        RobustScaler(),
        TransformedTargetRegressor(
            regressor=SVR(C=SVR_C, gamma=SVR_GAMMA, kernel='rbf', epsilon=0.0),
            transformer=QuantileTransformer(output_distribution='normal', n_quantiles=1000)
        )
    )

def cv_mae(splitter, **split_kwargs):
    oof = np.zeros(len(y))
    for tr_idx, va_idx in splitter.split(X, y, **split_kwargs):
        m = make_svr()
        m.fit(X.iloc[tr_idx], y[tr_idx])
        oof[va_idx] = m.predict(X.iloc[va_idx])
    return mean_absolute_error(y, oof)

kfold_mae = cv_mae(KFold(5, shuffle=True, random_state=RANDOM_STATE))
group_mae = cv_mae(GroupKFold(5), groups=groups)
const_mae = mean_absolute_error(y, np.full(len(y), np.median(y)))

print(f'상수(중앙값)         MAE : {const_mae:.4f}')
print(f'일반 KFold   SVR    MAE : {kfold_mae:.4f}   (기존 20번 노트북 CV = 0.1505, LB = 0.15291)')
print(f'근접쌍 GroupKFold SVR  MAE : {group_mae:.4f}')
print(f'차이(GroupKFold - KFold) : {group_mae - kfold_mae:+.4f}')

상수(중앙값)         MAE : 0.2493
일반 KFold   SVR    MAE : 0.1505   (기존 20번 노트북 CV = 0.1505, LB = 0.15291)
근접쌍 GroupKFold SVR  MAE : 0.2499
차이(GroupKFold - KFold) : +0.0995


## 결론

차이가 +0.0995. GroupKFold MAE(0.2499)는 상수 예측(0.2493)과 사실상 동일하다 -
즉 **SVR도 LGBM과 마찬가지로 진짜 예측력이 0에 가깝다.** raw14+bmi 피처와
stress_score 사이에는 (근접쌍을 통한 암기를 빼면) 실질적 관계가 없다.

그런데 실제 리더보드 점수는 0.15291로, 상수(0.25) 대비 크게 좋다. 이게 왜 가능한가:

SVR의 RBF 커널은 본질적으로 '가까운 훈련 샘플에 크게 의존하는' 지역적 모델이다.
test 행 중 48.4%가 train 안에 거의 동일한 쌍둥이를 갖고 있으므로(21번 노트북 확인),
SVR을 전체 train으로 학습시켜 test에 예측만 해도 커널 유사도가 자동으로 그 쌍둥이를
강하게 참조하게 된다. 즉 **21번 노트북의 명시적 매칭-복사와 원리적으로 같은 효과를
SVR이 암묵적으로, 조금 덜 완벽하게(0.1505 vs 21번의 예상 0.127) 재현하고 있는 것**이다.

결론: 이 데이터셋은 어떤 모델을 쓰든 - 명시적 매칭이든 SVR의 커널 유사도든 - train과
test 사이의 우연한(혹은 데이터 생성 과정의) 행 중복 구조를 이용하지 않고는 상수보다
나은 점수를 낼 방법이 사실상 없다. 대회 데이터 자체의 결함이며, 팀 단위의 피처
엔지니어링이나 모델 튜닝으로 해결할 수 있는 문제가 아니다.